# 02 · Limpieza de los datos

Partimos de los CSV de `data/raw` (se descargan con `python src/download_data.py`) y dejamos tres tablas limpias en `data/clean`:

- `jugadores.csv`: un jugador por fila.
- `equipos.csv`: un equipo por temporada.
- `estadisticas.csv`: una fila por jugador, temporada y equipo.

Solo usamos tres archivos del dataset: `Player Totals.csv`, `Player Career Info.csv` y `Team Abbrev.csv`.

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

totales = pd.read_csv(RAW_DIR / "Player Totals.csv")
jugadores_raw = pd.read_csv(RAW_DIR / "Player Career Info.csv")
equipos_raw = pd.read_csv(RAW_DIR / "Team Abbrev.csv")

print(totales.shape, jugadores_raw.shape, equipos_raw.shape)

(33780, 33) (5467, 11) (1848, 5)


## 1. Filtros

- Solo la liga **NBA** (el archivo incluye también la ABA y la BAA).
- Solo desde la temporada **1980**, la primera con línea de tres, hasta la **2026**. La 2027 es la temporada en curso y está incompleta.
- Los jugadores traspasados tienen una fila por equipo y otra con el total (`2TM`, `3TM`...). Quitamos las de total para no contar sus triples dos veces.
- Quitamos las filas sin minutos registrados.

In [2]:
print("Ligas:", totales["lg"].value_counts().to_dict())
print("Filas de total de traspasados:", totales["team"].str.contains("TM").sum())

stats = totales[
    (totales["lg"] == "NBA")
    & (totales["season"] >= 1980)
    & (totales["season"] <= 2026)   # la 2027 está a medias, la quitamos
    & (~totales["team"].str.contains("TM"))
    & (totales["mp"].notna())
].copy()

print("Filas antes:", len(totales), "| filas después:", len(stats))
print("Temporadas:", stats["season"].min(), "-", stats["season"].max())

Ligas: {'NBA': 31560, 'ABA': 1638, 'BAA': 582}
Filas de total de traspasados: 3052
Filas antes: 33780 | filas después: 23031
Temporadas: 1980 - 2026


Comprobación: los triples de Stephen Curry deben ser iguales sumando sus filas por equipo que usando el total del dataset.

In [3]:
nba80 = totales[(totales["lg"] == "NBA") & (totales["season"] >= 1980) & (totales["season"] <= 2026)]
con_totales = nba80[nba80["player_id"] == "curryst01"]["x3p"].sum()
sin_totales = stats[stats["player_id"] == "curryst01"]["x3p"].sum()
print("Curry, triples en todas las filas:", con_totales, "| sin filas de total:", sin_totales)

Curry, triples en todas las filas: 4248.0 | sin filas de total: 4248.0


## 2. Tabla de estadísticas

Añadimos los rebotes (ofensivos, defensivos y totales) para la hipótesis 2. Desde 1980 no tienen ningún nulo.

In [4]:
estadisticas = stats[[
    "player_id", "season", "team", "age", "pos", "g", "gs", "mp",
    "x3p", "x3pa", "fg", "fga", "pts",
    "orb", "drb", "trb"
]].rename(columns={
    "season": "temporada", "team": "equipo", "age": "edad", "pos": "posicion",
    "g": "partidos", "gs": "titular", "mp": "minutos",
    "x3p": "triples", "x3pa": "triples_intentados",
    "fg": "tiros_campo", "fga": "tiros_campo_intentados", "pts": "puntos",
    "orb": "rebotes_ofensivos", "drb": "rebotes_defensivos", "trb": "rebotes"
})

# en los 80 algunos jugadores no tienen triples registrados: los ponemos a 0
estadisticas[["triples", "triples_intentados"]] = estadisticas[["triples", "triples_intentados"]].fillna(0)

# los rebotes no deberían tener nulos desde 1980
print("Nulos en rebotes:", estadisticas[["rebotes_ofensivos", "rebotes_defensivos", "rebotes"]].isna().sum().sum())

# duplicados por jugador, temporada y equipo (debería salir 0)
print("Duplicados:", estadisticas.duplicated(["player_id", "temporada", "equipo"]).sum())
estadisticas.head()

Nulos en rebotes: 0
Duplicados: 0


,player_id,temporada,equipo,edad,posicion,partidos,titular,minutos,triples,triples_intentados,tiros_campo,tiros_campo_intentados,puntos,rebotes_ofensivos,rebotes_defensivos,rebotes
441,achiupr01,2026,SAC,26.0,C,73,57.0,1745.0,27.0,97.0,316,598,736,177.0,315.0,492.0
442,adamsst01,2026,HOU,32.0,C,32,11.0,730.0,0.0,0.0,70,139,187,145.0,131.0,276.0
443,adebaba01,2026,MIA,28.0,C,73,73.0,2365.0,127.0,400.0,506,1145,1468,149.0,583.0,732.0
445,agbajoc01,2026,TOR,25.0,SG,42,13.0,650.0,12.0,65.0,72,170,181,33.0,62.0,95.0
446,agbajoc01,2026,BRK,25.0,SG,20,0.0,323.0,22.0,63.0,50,110,133,13.0,33.0,46.0


## 3. Tabla de jugadores (solo los que aparecen en las estadísticas)

In [5]:
jugadores = jugadores_raw[jugadores_raw["player_id"].isin(estadisticas["player_id"])][[
    "player_id", "player", "pos", "ht_in_in", "wt", "birth_date", "colleges", "from", "to", "hof"
]].rename(columns={
    "player": "nombre", "pos": "posicion", "ht_in_in": "altura_pulgadas", "wt": "peso_libras",
    "birth_date": "fecha_nacimiento", "colleges": "universidades",
    "from": "primera_temporada", "to": "ultima_temporada", "hof": "hall_of_fame"
})

print(len(jugadores), "jugadores")
print("¿Falta algún jugador?", not estadisticas["player_id"].isin(jugadores["player_id"]).all())
jugadores.head()

3819 jugadores
¿Falta algún jugador? False


,player_id,nombre,posicion,altura_pulgadas,peso_libras,fecha_nacimiento,universidades,primera_temporada,ultima_temporada,hall_of_fame
826,silaspa01,Paul Silas,F-C,79.0,220.0,1943-07-12,Creighton,1965,1980,False
838,barryri01,Rick Barry,F,79.0,205.0,1944-03-28,Miami (FL),1966,1980,True
972,jacksph01,Phil Jackson,F-C,80.0,220.0,1945-09-17,University of North Dakota,1968,1980,True
988,monroea01,Earl Monroe,G,75.0,185.0,1944-11-21,Winston-Salem State,1968,1980,True
1038,fraziwa01,Walt Frazier,G,76.0,200.0,1945-03-29,Southern Illinois,1968,1980,True


## 4. Tabla de equipos (un equipo por temporada, porque algunos cambiaron de nombre o de ciudad)

In [6]:
equipos = equipos_raw[
    (equipos_raw["lg"] == "NBA") & (equipos_raw["season"] >= 1980)
][["season", "abbreviation", "team", "playoffs"]].rename(columns={
    "season": "temporada", "abbreviation": "equipo", "team": "nombre_equipo"
})

claves = estadisticas[["temporada", "equipo"]].drop_duplicates()
sin_equipo = claves.merge(equipos, on=["temporada", "equipo"], how="left")["nombre_equipo"].isna().sum()
print(len(equipos), "equipos-temporada | combinaciones sin equipo:", sin_equipo)
equipos.head()

1344 equipos-temporada | combinaciones sin equipo: 0


,temporada,equipo,nombre_equipo,playoffs
0,2027,ATL,Atlanta Hawks,False
1,2027,BOS,Boston Celtics,False
2,2027,BRK,Brooklyn Nets,False
3,2027,CHI,Chicago Bulls,False
4,2027,CHO,Charlotte Hornets,False


## 5. Guardar en data/clean

In [7]:
jugadores.to_csv(CLEAN_DIR / "jugadores.csv", index=False)
equipos.to_csv(CLEAN_DIR / "equipos.csv", index=False)
estadisticas.to_csv(CLEAN_DIR / "estadisticas.csv", index=False)

for archivo in sorted(CLEAN_DIR.glob("*.csv")):
    print(archivo.name)

equipos.csv
estadisticas.csv
jugadores.csv
